In [ ]:
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
import colorir as cl
import numpy as np

In [ ]:
sim_paths = []
for path in Path("../runs/speed/").iterdir():
    for replica_path in path.iterdir():
        sim_paths.append(replica_path / "cells")

In [ ]:
sims = pl.read_parquet(sim_paths, include_file_paths="path").with_columns(pl.col("path").str.split("/"))

In [ ]:
celldf = sims.with_columns(
    sim_replica=pl.col("path").list.get(-3).cast(int),
    gamma=20 - pl.col("path").list.get(-4).str.replace("energy-", "").cast(int),
)
celldf = celldf.with_columns(
    displ=(pl.col("center_x") ** 2 + pl.col("center_y") ** 2) ** 0.5
)
celldf

In [ ]:
grouppers = ["gamma", "sim_replica", "time"]
displdf = celldf.group_by(grouppers).agg(
    cluster_x=pl.col("center_x").mean(),
    cluster_y=pl.col("center_y").mean(),
    mean_displ=pl.col("displ").mean(),
    med_displ=pl.col("displ").median()
).sort(grouppers)
displdf

In [ ]:
# Cluster trajectories
px.line(
    displdf,
    x="cluster_x",
    y="cluster_y",
    color="gamma",
    line_dash="sim_replica"
)

In [ ]:
px.line(
    displdf,
    x="time",
    y="mean_displ",
    color="gamma",
    line_dash="sim_replica"
)

In [ ]:
repldf = displdf.group_by(["gamma", "time"]).agg(
    med_med=pl.col("med_displ").median(),
    med_mean=pl.col("med_displ").mean(),
    med_min=pl.col("med_displ").min(),
    med_max=pl.col("med_displ").max(),
    mean_mean=pl.col("mean_displ").mean(),
    mean_med=pl.col("mean_displ").median(),
    mean_min=pl.col("mean_displ").min(),
    mean_max=pl.col("mean_displ").max(),
).sort(["gamma", "time"])
repldf

In [ ]:
import random as rand

palette = cl.Grad(cl.StackPalette.load("carnival"), domain=[0, 20])
fig = go.Figure()
for gamma, group in repldf.group_by("gamma"):
    gamma = gamma[0]
    fig.add_trace(go.Scatter(
        x=group["time"],
        y=group["mean_med"],
        line_color=palette(gamma),
        name=gamma
    ))
    fig.add_trace(go.Scatter(
        x=pl.concat([group["time"], group["time"][::-1]]),
        y=pl.concat([group["mean_min"], group["mean_max"][::-1]]),
        fill="toself",
        line_color="rgba(0, 0, 0, 0)",
        fillcolor=palette(gamma),
        opacity=0.3,
        showlegend=False
    ))
fig

In [ ]:
concdf = displdf.filter(pl.col("time") > 5e6).group_by(["gamma"]).agg(
    med=pl.col("mean_displ").median(),
    min=pl.col("mean_displ").min(),
    max=pl.col("mean_displ").max(),
)
concdf

In [ ]:
fig = go.Figure([
    go.Scatter(
        x=concdf["gamma"],
        y=concdf["med"],
        line_color="black"
    ),
    go.Scatter(
        x=pl.concat([concdf["gamma"], concdf["gamma"][::-1]]),
        y=pl.concat([concdf["min"], concdf["max"][::-1]]),
        fill="toself",
        line_color="rgba(0, 0, 0, 0)",
        fillcolor="rgba(0, 0, 0, 0.2)"
    )
])
fig

In [ ]:
max_time = displdf["time"].max()
stdf = displdf\
    .group_by(["gamma", "sim_replica"])\
    .agg(
        stime=pl.col("time")\
            .filter(
                pl.col("mean_displ") <= pl.col("mean_displ").filter(pl.col("time") > 5e6).mean()
            )\
            .min(),
        ptime=pl.col("time")\
            .filter(
                pl.col("mean_displ") <= 180
            )\
            .min().fill_null(max_time)
    )
stdf

In [ ]:
aggs = ["gamma", "sim_replica", "time"]
filterdf = celldf\
    .join(stdf, on=["gamma", "sim_replica"])\
    .filter(
        pl.col("time") > 25e4,
        pl.col("time") <= pl.col("stime")
    )\
    .with_columns(
        mean_displ=pl.col("displ").mean().over(aggs),
        cluster_x=pl.col("center_x").mean().over(aggs),
        cluster_y=pl.col("center_y").mean().over(aggs),
    )\
    .sort(["gamma", "sim_replica", "index", "time"])
filterdf

In [ ]:
def circmean(expr):
    return pl.arctan2(expr.sin().sum(), expr.cos().sum())

def vel(colx, coly):
    xsq = pl.col(colx).diff() ** 2
    ysq = pl.col(coly).diff() ** 2
    return (xsq + ysq).sqrt()
    
cellveldf = filterdf\
    .sort("time")\
    .group_by(["gamma", "sim_replica", "index"])\
    .agg(
        vel=vel("center_x", "center_y").mean(),
        arc=circmean(pl.arctan2(
            pl.col("center_y").diff(), 
            pl.col("center_x").diff()
        ).diff())
)
cellveldf

In [ ]:
px.violin(cellveldf, x="gamma", y="arc").update_layout(width=600)

In [ ]:
px.box(cellveldf, x="gamma", y="vel").update_layout(width=600)

In [ ]:
def cos_align(colx, coly):
    colx = pl.col(colx)
    coly = pl.col(coly)
    seg_angles = pl.arctan2(
        coly.diff(), 
        colx.diff()
    )
    ref_angles = pl.arctan2(
        -coly, 
        -colx
    )
    return (seg_angles - ref_angles).cos()

clusterveldf = filterdf\
    .group_by(["gamma", "sim_replica", "time"])\
    .agg(
        pl.col("cluster_x").first(),
        pl.col("cluster_y").first()
    )\
    .sort(["gamma", "sim_replica", "time"])\
    .group_by(["gamma", "sim_replica"], maintain_order=True)\
    .agg(
        vel=vel("cluster_x", "cluster_y").mean(),
        arc=circmean(pl.arctan2(
            pl.col("cluster_y").diff(), 
            pl.col("cluster_x").diff()
        ).diff()),
        align=cos_align("cluster_x", "cluster_y").mean()
    )
clusterveldf

In [ ]:
# Gamma = 20 makes the cluster stutter, leading to lower alignment
px.scatter(clusterveldf, x="gamma", y="align", color="sim_replica").update_layout(width=600)

In [ ]:
# Two timesteps analysed at a time
dtdf = filterdf\
    .group_by(["gamma", "sim_replica", "time"])\
    .agg(
        pl.col("cluster_x").first(),
        pl.col("cluster_y").first()
    )\
    .filter(pl.col("gamma") % 10 == 0)\
    .sort(["gamma", "sim_replica", "time"])\
    .with_row_index()\
    .group_by(["gamma", "sim_replica", pl.col("index") // 2], maintain_order=True)\
    .agg(
        pl.len(),
        vel=vel("cluster_x", "cluster_y").mean(),
        align=cos_align("cluster_x", "cluster_y").mean()
    )
dtdf


In [ ]:
# Gamma 20 stutters
px.histogram(dtdf, x="align", color="gamma")

In [ ]:
px.scatter(clusterveldf, x="gamma", y="vel", color="sim_replica").update_layout(width=600)

In [ ]:
veldf = cellveldf.join(
    clusterveldf, 
    ["gamma", "sim_replica"],
    suffix="_cluster"
).group_by("gamma").agg(
    mean_vel=pl.col("vel").mean(),
    mean_vel_cluster=pl.col("vel_cluster").mean()
)
veldf

In [ ]:
# Non-linear relationship between cell speed and cluster speed
# Or it is linear but strength of CIL depends on gamma? 
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(
    x=veldf["gamma"],
    y=veldf["mean_vel"],
    mode="markers",
    name="cell"
))
fig.add_trace(go.Scatter(
    x=veldf["gamma"],
    y=veldf["mean_vel_cluster"],
    mode="markers",
    name="cluster"
), secondary_y=True)

In [ ]:
displ_cuts = np.linspace(0, (2 * 500 ** 2) ** 0.5, 6)

arcdf = celldf\
    .filter(pl.col("time") > 5e6)\
    .with_columns(
        q_displ=pl.col("displ").cut(displ_cuts)
    )\
    .sort("time")\
    .group_by(["gamma", "sim_replica", "index", "q_displ"])\
    .agg(
        arc=pl.arctan2(
            pl.col("center_y").diff(), 
            pl.col("center_x").diff()
        ).diff(),
        align=cos_align("center_x", "center_y")
    )\
    .explode(["arc", "align"])\
    .drop_nulls()\
    .sort(["gamma", "sim_replica", "q_displ"])
arcdf

In [ ]:
px.violin(
    arcdf,
    x="q_displ",
    y="arc",
    facet_row="gamma",
    violinmode="overlay",
    height=2000
).update_traces(width=1)

In [ ]:
px.violin(
    arcdf,
    x="q_displ",
    y="align",
    facet_row="gamma",
    violinmode="overlay",
    height=2000
).update_traces(width=1)